# 03 · 下游任务结果可视化

读取 `phenostate-retrieval` 的一个正式结果目录，不训练模型、不提取 embedding、不重新计算任务。
同一次评价保证各模型共用划分、有效孔位和候选资格。先在 val 固定方案，再查看 test。
公开预训练参考与本地训练模型的上游数据/计算预算不同，不能解释为同预算架构消融。

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
RESULT_DIR = Path(os.environ.get('PHENOSTATE_RESULTS_DIR', PROJECT_ROOT / 'outputs/downstream_val'))
PUBLIC_MODELS = {'OpenPhenom', 'MAE-L-8', 'MAE-G-8'}
TASKS = ('activity', 'same-guide', 'compound', 'cross-guide')
PRIMARY = {task: 'mrr' if task == 'cross-guide' else 'map' for task in TASKS}
if not (RESULT_DIR / 'protocol.json').is_file():
    raise FileNotFoundError(f'未找到正式评价结果：{RESULT_DIR}。先运行 phenostate-retrieval，或设置 PHENOSTATE_RESULTS_DIR。')
protocol = json.loads((RESULT_DIR / 'protocol.json').read_text(encoding='utf-8'))
coverage = json.loads((RESULT_DIR / 'coverage.json').read_text(encoding='utf-8'))
summary = pd.read_csv(RESULT_DIR / 'summary.csv')
models = summary.model.drop_duplicates().tolist()
results = {}
for model in models:
    for task in TASKS:
        path = RESULT_DIR / model / task / 'metrics.json'
        if path.is_file():
            results[model, task] = json.loads(path.read_text(encoding='utf-8'))

## 1. 协议、来源和覆盖率

先核对 train gallery、query split、预处理、相似度和各任务可评价率。
embedding 覆盖率与任务覆盖率的分母不同；缺失/不可评价记为 NA，不用 0 代替。
整板划分只规定当前评价孔位，不能撤销公开模型已经接触过 RxRx3 的预训练历史。

In [ ]:
display(pd.Series({key: protocol.get(key) for key in ('query_split', 'gallery_split', 'tasks', 'metric',
        'preprocessing', 'seed', 'n_bootstrap', 'n_permutations', 'n_null', 'split_scope')}, name='value').to_frame())
display(pd.Series({key: value for key, value in coverage.items()
                   if isinstance(value, (int, float))}, name='value').to_frame())
display(pd.DataFrame([{'model': name, 'public_reference': name in PUBLIC_MODELS,
        'dimension': info.get('embedding_dimension'), 'source_kind': info.get('source_kind'),
        'provenance': info.get('checkpoint_verification')}
        for name, info in protocol['models'].items()]))
columns = [c for c in ('model', 'task', 'score', 'n_evaluable_queries', 'n_groups', 'overall_query_coverage',
           'overall_group_coverage', 'replication_coverage', 'excluded_reasons') if c in summary]
display(summary[columns])

def points(ax, records, metric, bounds=(0, 1), random_field=None):
    for position, (label, stats) in enumerate(records):
        score = stats.get(metric)
        if score is None or not np.isfinite(score):
            ax.text(0.02, position, 'NA', va='center')
            continue
        interval = stats.get('ci95', {}).get(metric)
        if interval is not None and np.isfinite(interval).all():
            ax.hlines(position, interval[0], interval[1], color='tab:blue', linewidth=2)
        ax.scatter(score, position, marker='^' if label.split(' / ')[0] in PUBLIC_MODELS else 'o', color='tab:blue', clip_on=False)
        if random_field and stats.get(random_field) is not None:
            ax.scatter(stats[random_field], position, marker='x', color='gray')
    ax.set_yticks(range(len(records)), [label for label, _ in records])
    ax.set_ylim(-0.5, max(len(records) - 0.5, 0.5))
    if bounds is not None:
        ax.set_xlim(*bounds)
    ax.invert_yaxis()
    ax.set_xlabel(metric)
    ax.grid(axis='x', alpha=0.25)
    if not records:
        ax.text(0.5, 0.5, 'Not evaluated', transform=ax.transAxes, ha='center')

## 2. 主指标与组级 95% CI

activity、same-guide、compound 看 mAP，cross-guide 看 MRR。四个面板分别解释，不合成为一个总分。
三角点为配置中的公开参考，圆点为其他模型；灰色叉为协议提供的随机基线。
CI 直接读取扰动组 bootstrap；它不是训练随机种子的不确定性。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, max(7, len(models) * 0.8)), squeeze=False)
for ax, task in zip(axes.flat, TASKS):
    records = [(name, results[name, task]['summary']) for name in models if (name, task) in results]
    points(ax, records, PRIMARY[task], random_field='random_mrr' if task == 'cross-guide' else 'random_ap')
    ax.set_title(task)
plt.tight_layout()
plt.show()

## 3. Recall@K、重复一致性与显著活性

Recall 是找回全部正例的比例，不是 Hit@K。percent replicating、percent active 已按 0–100 存储，不再乘 100。
显著活性按 CRISPR/COMPOUND 分开，受严格对照匹配、重复数、置换次数及 BH 阈值影响。
energy distance 另看诊断表，不能与 mAP/MRR 放在同一评分轴上。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), squeeze=False)
for ax, task in zip(axes.flat, ('same-guide', 'compound', 'cross-guide')):
    for name in models:
        if (name, task) not in results:
            continue
        stats = results[name, task]['summary']
        ks = protocol['ks']
        ys = [np.nan if stats.get(f'recall@{k}') is None else stats[f'recall@{k}'] for k in ks]
        ax.plot(ks, ys, 'o-', label=name)
    ax.set(title=task, xlabel='K', ylabel='Recall@K', ylim=(0, 1))
    ax.set_xticks(protocol['ks'])
    ax.grid(alpha=0.2)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, max(7, len(models) * 0.8)), squeeze=False)
for ax, task in zip(axes[0], ('same-guide', 'compound')):
    records = [(name, results[name, task]['summary']) for name in models if (name, task) in results]
    points(ax, records, 'percent_replicating', bounds=(0, 100))
    ax.set_title(task + ' / replicating (%)')
activity_rows = []
for ax, kind in zip(axes[1], ('CRISPR', 'COMPOUND')):
    records = []
    for name in models:
        stats = results.get((name, 'activity'), {}).get('summary', {}).get('by_kind', {}).get(kind)
        if stats:
            records.append((name, stats))
            activity_rows.append({'model': name, 'type': kind, **{key: stats.get(key) for key in
                                 ('n_groups', 'n_tests', 'percent_active', 'percent_energy_active', 'energy_distance')}})
    points(ax, records, 'percent_active', bounds=(0, 100))
    ax.set_title(kind + ' / AP-significant activity (%)')
plt.tight_layout()
plt.show()
display(pd.DataFrame(activity_rows))

## 4. 各扰动组表现分布

ECDF 使用已保存的分组 AP/MRR，每个扰动组一票；不把重复 well 当成独立生物学重复。
曲线越偏右表示较多组得分较高，但应结合覆盖率与主指标 CI。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
for ax, task in zip(axes.flat, TASKS):
    field = 'mrr' if task == 'cross-guide' else 'ap'
    for name in models:
        path = RESULT_DIR / name / task / 'groups.csv'
        if not path.is_file():
            continue
        frame = pd.read_csv(path)
        if field not in frame:
            continue
        values = np.sort(pd.to_numeric(frame[field], errors='coerce').dropna().to_numpy())
        if len(values):
            ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', label=f'{name} (n={len(values)})')
    ax.set(title=task, xlabel=field, ylabel='Fraction of perturbation groups', xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.2)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. 配对差值：相对于一个参考模型

只读 `paired_differences.csv` 的配对扰动 bootstrap CI，不能用两个独立 CI 的端点相减。
正值代表 other − baseline > 0。不同任务分面展示；单模型运行时无配对结果。

In [ ]:
BASELINE_MODEL = models[0]  # 可改为 'OpenPhenom' 或某个自训练模型名。
paired = pd.read_csv(RESULT_DIR / 'paired_differences.csv')
fig, axes = plt.subplots(2, 2, figsize=(12, max(7, len(models) * 0.7)), squeeze=False)
for ax, task in zip(axes.flat, TASKS):
    records = []
    for row in paired.loc[paired.task.eq(task)].itertuples():
        if BASELINE_MODEL not in (row.model_a, row.model_b):
            continue
        sign = 1 if row.model_a == BASELINE_MODEL else -1
        other = row.model_b if sign == 1 else row.model_a
        lo, hi = row.ci95_lower, row.ci95_upper
        interval = [lo, hi] if sign == 1 else [-hi, -lo]
        records.append((other, {'delta': sign * row.delta_b_minus_a, 'ci95': {'delta': interval}}))
    points(ax, records, 'delta', bounds=None)
    ax.axvline(0, linestyle='--', color='gray')
    ax.set_title(task + ' / other - ' + BASELINE_MODEL)
plt.tight_layout()
plt.show()

## 6. 选择模型、任务和扰动组查看诊断

查看每组 AP/MRR、query 排除原因、activity 匹配 block，以及 PR 的观测相似度与 null 95% 阈值。
可填写 `GROUP_LABEL` 筛选单组。表型活性不是独立的处理预测准确率；无有效对照或 null 的结果不记为阴性。

In [ ]:
MODEL = models[0]
TASK = next(task for task in TASKS if (MODEL, task) in results)
GROUP_LABEL = ''
DISPLAY_ROWS = 20
result = results[MODEL, TASK]
display(pd.Series(result['summary'], name='value').to_frame())
tables = {}
for filename in ('groups.csv', 'queries.csv', 'blocks.csv', 'replicates.csv'):
    path = RESULT_DIR / MODEL / TASK / filename
    if not path.is_file():
        continue
    frame = pd.read_csv(path)
    if GROUP_LABEL and 'perturbation_label' in frame:
        frame = frame.loc[frame.perturbation_label.eq(GROUP_LABEL)]
    tables[filename] = frame
    field = 'mrr' if TASK == 'cross-guide' else 'ap'
    if filename == 'groups.csv' and field in frame:
        frame = frame.sort_values(field)
    print(filename, '| rows:', len(frame))
    display(frame.head(DISPLAY_ROWS))
replicates = tables.get('replicates.csv', pd.DataFrame())
if {'median_similarity', 'null_q95'} <= set(replicates):
    valid = replicates[['median_similarity', 'null_q95']].dropna()
    if not valid.empty:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(valid.null_q95, valid.median_similarity, s=15, alpha=0.6)
        low, high = valid.to_numpy().min(), valid.to_numpy().max()
        ax.plot([low, high], [low, high], '--', color='gray')
        ax.set(xlabel='Matched null 95% threshold', ylabel='Observed replicate similarity', title=MODEL + ' / ' + TASK)
        plt.tight_layout()
        plt.show()